# Manager change analysis: Man United and Real Madrid, 2026-27

**How to use this notebook:** click **Run All** at the top. Cells run from top to bottom, and each section builds on the one before it. Don't run cells out of order.

| Section | What it does |
|---|---|
| 1. Setup | Imports and the key dates |
| 2. Load data | Downloads this season's Premier League and La Liga results |
| 3. Helper functions | Reusable code (reshape a team's matches, calculate ratios) |
| 4. Build the groups | Carrick, Pochettino and Madrid (league and Champions League) |
| 5. Sanity checks | Confirms the numbers match what we know |
| 6. Summary table | The main comparison |
| 7. Charts | Saves the PNGs used in the README |
| 8. Save files | Writes the clean CSVs |

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PL_URL = "https://www.football-data.co.uk/mmz4281/2627/E0.csv"
LALIGA_URL = "https://www.football-data.co.uk/mmz4281/2627/SP1.csv"

CARRICK_END = "2026-11-12"        # Carrick sacked
POCHETTINO_START = "2026-11-18"   # Pochettino appointed

## 2. Load data
League results and shot counts come from football-data.co.uk. The Champions League files (`cl_united_carrick.csv`, `cl_madrid_mourinho.csv`) must be in the same folder as this notebook.

In [ ]:
def load_league(url):
    df = pd.read_csv(url)
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    return df.dropna(subset=["Date"])

pl = load_league(PL_URL)
laliga = load_league(LALIGA_URL)

print("Premier League matches loaded:", len(pl), "| latest:", pl["Date"].max().date())
print("La Liga matches loaded:", len(laliga), "| latest:", laliga["Date"].max().date())

## 3. Helper functions
- `team_view` turns the home/away table into one team's point of view (goals for, goals against, shots for, shots against).
- `ratios` calculates shot accuracy and goals per shot on target, for and against. It skips matches with no shot data.
- `league_ratios` gives the league average to compare against.

In [ ]:
def team_view(df, team):
    d = df[(df["HomeTeam"] == team) | (df["AwayTeam"] == team)].copy()
    home = d["HomeTeam"] == team
    d["Venue"] = np.where(home, "Home", "Away")
    d["Opponent"] = np.where(home, d["AwayTeam"], d["HomeTeam"])
    d["GF"] = np.where(home, d["FTHG"], d["FTAG"])
    d["GA"] = np.where(home, d["FTAG"], d["FTHG"])
    d["Shots"] = np.where(home, d["HS"], d["AS"])
    d["ShotsAgainst"] = np.where(home, d["AS"], d["HS"])
    d["SoT"] = np.where(home, d["HST"], d["AST"])
    d["SoTAgainst"] = np.where(home, d["AST"], d["HST"])
    d["Corners"] = np.where(home, d["HC"], d["AC"])
    d["Fouls"] = np.where(home, d["HF"], d["AF"])
    d["Points"] = np.select([d["GF"] > d["GA"], d["GF"] == d["GA"]], [3, 1], default=0)
    keep = ["Date", "Opponent", "Venue", "GF", "GA", "Points", "Shots",
            "ShotsAgainst", "SoT", "SoTAgainst", "Corners", "Fouls"]
    return d[keep].sort_values("Date").reset_index(drop=True)


def ratios(tv):
    s = tv.dropna(subset=["Shots", "SoT"])
    t = s[["GF", "GA", "Shots", "ShotsAgainst", "SoT", "SoTAgainst"]].sum()

    def div(a, b):
        return round(a / b, 3) if b else np.nan

    return {
        "Games": len(tv),
        "Games with shot data": len(s),
        "Points": int(tv["Points"].sum()),
        "Goals for": int(tv["GF"].sum()),
        "Goals against": int(tv["GA"].sum()),
        "Shot accuracy": div(t["SoT"], t["Shots"]),
        "Goals per SoT": div(t["GF"], t["SoT"]),
        "Opp shot accuracy": div(t["SoTAgainst"], t["ShotsAgainst"]),
        "Opp goals per SoT": div(t["GA"], t["SoTAgainst"]),
    }


def league_ratios(df):
    return {
        "Shot accuracy": round((df["HST"].sum() + df["AST"].sum()) / (df["HS"].sum() + df["AS"].sum()), 3),
        "Goals per SoT": round((df["FTHG"].sum() + df["FTAG"].sum()) / (df["HST"].sum() + df["AST"].sum()), 3),
    }

## 4. Build the groups
Each group is one team in one competition and period. Keeping them separate makes it clear what is being compared.

In [ ]:
# Man United
united = team_view(pl, "Man United")
united_carrick_league = united[united["Date"] < CARRICK_END].assign(Competition="Premier League")
united_pochettino_league = united[united["Date"] >= POCHETTINO_START].assign(Competition="Premier League")
united_cl = pd.read_csv("cl_united_carrick.csv", parse_dates=["Date"])
united_carrick_all = pd.concat([united_carrick_league, united_cl]).sort_values("Date").reset_index(drop=True)

# Real Madrid (whole 2026-27 season is the Mourinho era)
madrid_league = team_view(laliga, "Real Madrid").assign(Competition="La Liga")
madrid_cl = pd.read_csv("cl_madrid_mourinho.csv", parse_dates=["Date"])
madrid_all = pd.concat([madrid_league, madrid_cl]).sort_values("Date").reset_index(drop=True)

## 5. Sanity checks
`OK` means the number matches what we know. `CHECK` means something is off, so look at it before trusting the results below.

In [ ]:
def check(label, got, expected):
    status = "OK   " if got == expected else "CHECK"
    print(f"{status} {label}: got {got}, expected {expected}")

check("Carrick league games", len(united_carrick_league), 10)
check("Carrick league points", int(united_carrick_league["Points"].sum()), 8)
check("United CL games (Carrick)", len(united_cl), 4)
check("Madrid CL games", len(madrid_cl), 4)

print()
print("Madrid La Liga games / points:", len(madrid_league), "/", int(madrid_league["Points"].sum()), "(compare with the La Liga table)")
print("Pochettino league games so far:", len(united_pochettino_league))

## 6. Summary table
Compare each team with **its own league's average**, not one shared number. The Champions League has no average here, so compare the two teams with each other and treat the gaps cautiously.

In [ ]:
groups = {
    "United Carrick league": united_carrick_league,
    "United Carrick CL": united_cl,
    "United Carrick all": united_carrick_all,
    "United Pochettino league": united_pochettino_league,
    "Madrid league": madrid_league,
    "Madrid CL": madrid_cl,
    "Madrid all": madrid_all,
}

summary = pd.DataFrame({name: ratios(g) for name, g in groups.items()})
display(summary)

print("Premier League average:", league_ratios(pl))
print("La Liga average:       ", league_ratios(laliga))

**Reading the table**
- Small samples: 4 Champions League games per team (3 for Madrid's shots), and Pochettino's group is tiny. Treat gaps as leads, not conclusions.
- League numbers come from football-data.co.uk and Champions League numbers from a different provider, so shot definitions may differ slightly.
- Madrid vs Inter has no shot data because the source's figures didn't add up. Its score still counts in goals and points.

## 7. Charts

In [ ]:
c = united_carrick_league

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(c))
ax.bar([i - 0.2 for i in x], c["Shots"], width=0.4, label="Shots for")
ax.bar([i + 0.2 for i in x], c["ShotsAgainst"], width=0.4, label="Shots against")
ax.set_xticks(list(x))
ax.set_xticklabels(c["Opponent"], rotation=45, ha="right")
ax.set_title("Manchester United shots for vs against, first 10 league games")
ax.legend()
plt.tight_layout()
plt.savefig("shots_for_vs_against.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(c["Opponent"], c["GF"].cumsum(), marker="o", label="Goals scored (cumulative)")
ax.plot(c["Opponent"], c["GA"].cumsum(), marker="o", label="Goals conceded (cumulative)")
ax.set_xticks(range(len(c)))
ax.set_xticklabels(c["Opponent"], rotation=45, ha="right")
ax.set_title("Cumulative goals scored vs conceded")
ax.legend()
plt.tight_layout()
plt.savefig("cumulative_goals.png", dpi=150)
plt.show()

## 8. Save files

In [ ]:
united_carrick_league.to_csv("carrick_era_clean.csv", index=False)
united_pochettino_league.to_csv("pochettino_era_clean.csv", index=False)
madrid_league.to_csv("madrid_league_clean.csv", index=False)
print("Saved: carrick_era_clean.csv, pochettino_era_clean.csv, madrid_league_clean.csv")